In [ ]:
# Install Kaggle API
!pip install kaggle

# Create directory
!mkdir -p ~/.kaggle

# Copy kaggle.json (must be uploaded first)
!cp /content/kaggle.json ~/.kaggle/

# Change permissions
!chmod 600 ~/.kaggle/kaggle.json

print("Kaggle API Successfully Configured!")

Kaggle API Successfully Configured!


In [ ]:
!git clone https://github.com/ultralytics/ultralytics
%cd ultralytics

Cloning into 'ultralytics'...
remote: Enumerating objects: 75946, done.
remote: Counting objects: 100% (354/354), done.
remote: Compressing objects: 100% (196/196), done.
remote: Total 75946 (delta 253), reused 200 (delta 158), pack-reused 75592 (from 3)
Receiving objects: 100% (75946/75946), 40.80 MiB | 26.41 MiB/s, done.
Resolving deltas: 100% (57029/57029), done.
/content/ultralytics


In [ ]:
!pip install -e .
import torch
print("GPU:", torch.cuda.get_device_name(0))

Obtaining file:///content/ultralytics
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for ultralytics (pyproject.toml) ... done
  Created wheel for ultralytics: filename=ultralytics-8.3.234-0.editable-py3-none-any.whl size=23169 sha256=d1fab720e1fff36755fea795f3c0ed4eac3100c416f2e2b945c5887a3674a328
  Stored in directory: /tmp/pip-ephem-wheel-cache-u0lahbeh/wheels/60/e0/59/e2f034f296abbdca5c21e3f5be76b9ca685f13c7bd17f8b58c
Successfully built ultralytics
GPU: Tesla T4


In [ ]:
import os, shutil

print("🧹 Cleaning datasets directory...")
if os.path.exists('/content/datasets'):
    shutil.rmtree('/content/datasets')

os.makedirs('/content/datasets')

%cd /content/datasets

🧹 Cleaning datasets directory...
/content/datasets


In [ ]:
print("⬇️ Downloading Brain Tumor Dataset...")
!kaggle datasets download -d pkdarabi/medical-image-dataset-brain-tumor-detection

⬇️ Downloading Brain Tumor Dataset...
Dataset URL: https://www.kaggle.com/datasets/pkdarabi/medical-image-dataset-brain-tumor-detection
License(s): Attribution 4.0 International (CC BY 4.0)
 51% 150M/297M [00:00<00:00, 1.57GB/s]
100% 297M/297M [00:03<00:00, 99.4MB/s]


In [ ]:
print("📦 Unzipping...")
!unzip -q medical-image-dataset-brain-tumor-detection.zip
!rm medical-image-dataset-brain-tumor-detection.zip

📦 Unzipping...


In [ ]:
import os

print("🔍 Locating data.yaml...")

yaml_path = None

for root, dirs, files in os.walk('/content/datasets'):
    if 'data.yaml' in files:
        yaml_path = os.path.join(root, 'data.yaml')
        break

if yaml_path is None:
    raise FileNotFoundError("❌ data.yaml not found!")

dataset_root = os.path.abspath(os.path.dirname(yaml_path))

yaml_content = f"""
path: {dataset_root}
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['glioma', 'meningioma', 'pituitary']
"""

with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print("✅ FIXED data.yaml at:", yaml_path)
print("📍 Dataset root:", dataset_root)

🔍 Locating data.yaml...
✅ FIXED data.yaml at: /content/datasets/BrainTumor/BrainTumorYolov11/data.yaml
📍 Dataset root: /content/datasets/BrainTumor/BrainTumorYolov11


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MSAF(nn.Module):
    def __init__(self, c1, c2=None, reduction=16):
        super().__init__()
        if c2 is None:
            c2 = c1
        if c1 != c2:
            c1 = c2

        # Split into multiple parts and perform attention on each part
        self.split1 = nn.Conv2d(c1, c1 // 2, 1)
        self.split2 = nn.Conv2d(c1, c1 // 2, 1)

        # Attention mechanism for each split
        self.attn1 = nn.Sequential(
            nn.Conv2d(c1 // 2, c1 // (2 * reduction), 1),
            nn.ReLU(),
            nn.Conv2d(c1 // (2 * reduction), c1 // 2, 1),
            nn.Sigmoid()
        )

        self.attn2 = nn.Sequential(
            nn.Conv2d(c1 // 2, c1 // (2 * reduction), 1),
            nn.ReLU(),
            nn.Conv2d(c1 // (2 * reduction), c1 // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Split the input into two parts
        part1 = self.split1(x)
        part2 = self.split2(x)

        # Apply attention to each part
        attn1 = self.attn1(part1)
        attn2 = self.attn2(part2)

        # Fuse the attended parts
        fused = torch.cat([attn1 * part1, attn2 * part2], dim=1)

        return fused

In [ ]:
# YOLOv8n-MSAF Custom Architecture
custom_model_yaml = """
nc: 3  # Classes: Glioma, Meningioma, Pituitary
scales:
  n: [0.33, 0.25, 1024]  # Model depth, width, max channels

backbone:
  - [-1, 1, Conv, [64, 3, 2]]  # 0
  - [-1, 1, Conv, [128, 3, 2]]  # 1
  - [-1, 3, C2f, [128, True]]
  - [-1, 1, Conv, [256, 3, 2]]  # 3
  - [-1, 6, C2f, [256, True]]
  - [-1, 1, Conv, [512, 3, 2]]  # 5
  - [-1, 6, C2f, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]]  # 7
  - [-1, 3, C2f, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]]  # 9

# Add the new MSAF layer after SPPF (replacing FCANet with MSAF)
  - [-1, 1, MSAF, [256]]  # 10 (MSAF layer)

head:
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 6], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 4], 1, Concat, [1]]
  - [-1, 3, C2f, [256]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]]
  - [-1, 3, C2f, [512]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]]
  - [-1, 3, C2f, [1024]]
  - [[16, 19, 22], 1, Detect, [nc]]  # Detect layer
"""

# Save the YAML file in the ultralytics directory
with open('/content/ultralytics/yolov8n-msaf.yaml', 'w') as f:
    f.write(custom_model_yaml)

print("✅ Custom model architecture saved to /content/ultralytics/yolov8n-msaf.yaml")


✅ Custom model architecture saved to /content/ultralytics/yolov8n-msaf.yaml


In [ ]:
import re
import os

# Path to the YOLOv8 code (tasks.py)
tasks_path = "/content/ultralytics/ultralytics/nn/tasks.py"

# MSAF code to inject
msaf_code = """
# --- MSAF LAYER START ---
import torch
import torch.nn as nn

class MSAF(nn.Module):
    def __init__(self, c1, c2=None, reduction=16):
        super().__init__()
        if c2 is None:
            c2 = c1
        if c1 != c2:
            c1 = c2

        # Split into multiple parts and perform attention on each part
        self.split1 = nn.Conv2d(c1, c1 // 2, 1)
        self.split2 = nn.Conv2d(c1, c1 // 2, 1)

        # Attention mechanism for each split
        self.attn1 = nn.Sequential(
            nn.Conv2d(c1 // 2, c1 // (2 * reduction), 1),
            nn.ReLU(),
            nn.Conv2d(c1 // (2 * reduction), c1 // 2, 1),
            nn.Sigmoid()
        )

        self.attn2 = nn.Sequential(
            nn.Conv2d(c1 // 2, c1 // (2 * reduction), 1),
            nn.ReLU(),
            nn.Conv2d(c1 // (2 * reduction), c1 // 2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        # Split the input into two parts
        part1 = self.split1(x)
        part2 = self.split2(x)

        # Apply attention to each part
        attn1 = self.attn1(part1)
        attn2 = self.attn2(part2)

        # Fuse the attended parts
        fused = torch.cat([attn1 * part1, attn2 * part2], dim=1)

        return fused
# --- MSAF LAYER END ---
"""

# Inject MSAF into the tasks.py file
with open(tasks_path, 'r') as f:
    content = f.read()

new_content = content

# Find where to inject the new code
if "class MSAF" not in content:
    match = re.search(r"^class \w+", content, re.MULTILINE)
    if match:
        idx = match.start()
        new_content = content[:idx] + msaf_code + "\n\n" + content[idx:]

# Register MSAF in the model parser
# Check if the string exists before trying to replace it.
if "if m in (" in new_content:
    new_content = re.sub(r"if m in \(", "if m in (MSAF, ", new_content, count=1)

# Write the modified content back to the tasks.py file
with open(tasks_path, 'w') as f:
    f.write(new_content)

print("✅ MSAF Class Injection Complete.")


✅ MSAF Class Injection Complete.


In [ ]:
%cd /content/ultralytics
!yolo detect train \
data=/content/datasets/BrainTumor/BrainTumorYolov11/data.yaml \
model=/content/ultralytics/yolov8n-msaf.yaml \
epochs=50 \
imgsz=640 \
batch=16 \
name=MSAF_YOLOv8n_Tumor

/content/ultralytics
Ultralytics 8.3.234 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/datasets/BrainTumor/BrainTumorYolov11/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/ultralytics/yolov8n-msaf.yaml, momentum=0.937, mosaic=1.0, multi_scale=False, name=MSAF_YOLOv8n_Tumor, nbs=64, nms=False, opset=None, optimize=F